# Journey to LLM and prompt engineering

## The Chat Format

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

### Setup

In [4]:
import os
import openai
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.getenv('OPENAI_API_KEY')


In [6]:
from openai import OpenAI

def get_completion(prompt, model="gpt-4o-mini"):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content


# Initialize the client
client = OpenAI(api_key=openai.api_key)

def get_completion_from_messages(messages, model="gpt-4o-mini", temperature=0):
    # The new syntax: client.chat.completions.create
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    
    # Accessing the content is now done via dot notation, not a dictionary
    return response.choices[0].message.content

messages = [
    {'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},
    {'role':'user', 'content':'tell me a joke'},
    {'role':'assistant', 'content':'Why did the chicken cross the road'},
    {'role':'user', 'content':'I don\'t know'}
]

response = get_completion_from_messages(messages, temperature=1)
print(response)


To get to the other side, my good fellow! Oh, how mirthful is this jest!


#### "temperature=1" means maximum diverse and creative!  Each time the response would be totally different.

In [7]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]

response = get_completion_from_messages(messages, temperature=1)
print(response)


Hello Isa! It's nice to meet you. How can I assist you today?


### Didn't mention names.

In [8]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]

response = get_completion_from_messages(messages, temperature=1)
print(response)


I'm sorry, but I can't remember your name. However, you can tell me your name and I'll gladly address you by it.


In [9]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]

response = get_completion_from_messages(messages, temperature=1)
print(response)


Your name is Isa.


## Zero-shot prompting

In [10]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Classify the text into positive, neutral or negative:\
Text: That shot selection was awesome.\
Classification:'}  ]

response = get_completion_from_messages(messages, temperature=1)
print(response)


Positive


## Few-shot prompting

In [11]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Text: Today the weather is fantastic\
Classification: Pos\
Text: The furniture is small.\
Classification: Neu\
Text: I do not like your attitude\
Classification: Neg\
Text: That shot selection was awful\
Classification:'}  ]

response = get_completion_from_messages(messages, temperature=1)
print(response)


Neg


## Chain-of-thought prompting

In [12]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'The cafeteria had 23 apples. \
If they used 20 to make lunch and bought 6 more, how many apples do they have? \
Let us think step by step.'}  ]

response = get_completion_from_messages(messages, temperature=1)
print(response)


Sure, let's work through it step by step.

1. The cafeteria started with 23 apples.
2. They used 20 apples to make lunch, so they have 23 - 20 = 3 apples left.
3. Then they bought 6 more apples.
4. Adding the 3 remaining apples to the 6 new apples gives us 3 + 6 = 9 apples.

So, the cafeteria now has 9 apples in total.


In [13]:
delimiter = "####"
system_message = f"""
Follow these steps to answer the customer queries.
The customer query will be delimited with four hashtags,\
i.e. {delimiter}. 

Step 1:{delimiter} First decide whether the user is \
asking a question about a specific product or products. \
Product cateogry doesn't count. 

Step 2:{delimiter} If the user is asking about \
specific products, identify whether \
the products are in the following list.
All available products: 
1. Product: TechPro Ultrabook
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-UB100
   Warranty: 1 year
   Rating: 4.5
   Features: 13.3-inch display, 8GB RAM, 256GB SSD, Intel Core i5 processor
   Description: A sleek and lightweight ultrabook for everyday use.
   Price: $799.99

2. Product: BlueWave Gaming Laptop
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-GL200
   Warranty: 2 years
   Rating: 4.7
   Features: 15.6-inch display, 16GB RAM, 512GB SSD, NVIDIA GeForce RTX 3060
   Description: A high-performance gaming laptop for an immersive experience.
   Price: $1199.99

3. Product: PowerLite Convertible
   Category: Computers and Laptops
   Brand: PowerLite
   Model Number: PL-CV300
   Warranty: 1 year
   Rating: 4.3
   Features: 14-inch touchscreen, 8GB RAM, 256GB SSD, 360-degree hinge
   Description: A versatile convertible laptop with a responsive touchscreen.
   Price: $699.99

4. Product: TechPro Desktop
   Category: Computers and Laptops
   Brand: TechPro
   Model Number: TP-DT500
   Warranty: 1 year
   Rating: 4.4
   Features: Intel Core i7 processor, 16GB RAM, 1TB HDD, NVIDIA GeForce GTX 1660
   Description: A powerful desktop computer for work and play.
   Price: $999.99

5. Product: BlueWave Chromebook
   Category: Computers and Laptops
   Brand: BlueWave
   Model Number: BW-CB100
   Warranty: 1 year
   Rating: 4.1
   Features: 11.6-inch display, 4GB RAM, 32GB eMMC, Chrome OS
   Description: A compact and affordable Chromebook for everyday tasks.
   Price: $249.99

Step 3:{delimiter} If the message contains products \
in the list above, list any assumptions that the \
user is making in their \
message e.g. that Laptop X is bigger than \
Laptop Y, or that Laptop Z has a 2 year warranty.

Step 4:{delimiter}: If the user made any assumptions, \
figure out whether the assumption is true based on your \
product information. 

Step 5:{delimiter}: First, politely correct the \
customer's incorrect assumptions if applicable. \
Only mention or reference products in the list of \
5 available products, as these are the only 5 \
products that the store sells. \
Answer the customer in a friendly tone.

Use the following format:
Step 1:{delimiter} <step 1 reasoning>
Step 2:{delimiter} <step 2 reasoning>
Step 3:{delimiter} <step 3 reasoning>
Step 4:{delimiter} <step 4 reasoning>
Response to user:{delimiter} <response to customer>

Make sure to include {delimiter} to separate every step.
"""


In [14]:
user_message = f"""
by how much is the BlueWave Chromebook more expensive \
than the TechPro Desktop"""

messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message}{delimiter}"},  
] 

response = get_completion_from_messages(messages)
print(response)


Step 1:#### The user is comparing the prices of two specific products.
Step 2:#### The products being compared are:
- BlueWave Chromebook: Price $249.99
- TechPro Desktop: Price $999.99
Step 3:#### The user assumes that the BlueWave Chromebook is more expensive than the TechPro Desktop.
Step 4:#### The assumption made by the user is incorrect. The TechPro Desktop is actually more expensive than the BlueWave Chromebook.
Response to user:#### The TechPro Desktop is actually more expensive than the BlueWave Chromebook. The TechPro Desktop is priced at $999.99, while the BlueWave Chromebook is priced at $249.99.


In [15]:
user_message = f"""
do you sell tvs"""
messages =  [  
{'role':'system', 
 'content': system_message},    
{'role':'user', 
 'content': f"{delimiter}{user_message}{delimiter}"},  
] 

response = get_completion_from_messages(messages)
print(response)


Step 1:#### The user is asking about a specific product category (TVs) and not a specific product.
Step 2:#### N/A
Step 3:#### N/A
Step 4:#### N/A
Response to user:#### We currently do not sell TVs. Our store specializes in computers and laptops. If you have any questions about our available products, feel free to ask!
